# Tái lập dữ liệu MassSpecGym cho `ms-pred` / MARASON trên Kaggle

Notebook dựng lại dataset từ repo `Giamyy/ms-pred` (fork của coleygroup/ms-pred) theo README:

1. Cài môi trường bằng `uv sync --extra cpu --python 3.11` theo `pyproject.toml`.
2. Build `labels.tsv`, `spec_files.hdf5`, `splits/split.tsv` bằng **script chính thức**
   `data_scripts/create_msg_simulation_dataset.py` trực tiếp từ MassSpecGym 1.5 trên Hugging Face.
3. Chạy MAGMa (`magma_outputs/`) và gán subformula (`subformulae/no_subform.hdf5`) như `data_scripts/dag/run_magma.sh`.
4. (Tuỳ chọn) thêm bảng ứng viên retrieval.
5. Lưu kết quả vào `/kaggle/working` để tạo Kaggle Dataset.

**Vì sao chia shard:** một phiên Kaggle tối đa 12 giờ, còn MAGMa cho 119,029 phổ mất khoảng 24 giờ với 4 CPU.
MAGMa xử lý từng phổ độc lập, nên chạy từng phần rồi gộp cho cùng kết quả như chạy một lần.

| `STAGE` | Làm gì |
|---|---|
| `"shard"` | Build dataset, rồi chạy MAGMa + subformula cho phần `SHARD_ID` trong `NUM_SHARDS` phần |
| `"merge"` | Build dataset, gộp kết quả các shard (Add Input), thêm bảng retrieval, đóng gói |
| `"all"` | Làm hết trong một phiên. Chỉ dùng để chạy thử với `SUBSET_FRAC` nhỏ |

**Cách chạy dữ liệu đầy đủ (4 shard, mỗi shard khoảng 6 giờ):**
1. Tạo 4 bản sao notebook (`msg-shard-0` … `msg-shard-3`): `STAGE = "shard"`, `SHARD_ID` = 0..3,
   *Accelerator: None*, *Internet: On* → *Save Version → Save & Run All* (chạy song song được).
2. Khi cả 4 xong: notebook `msg-merge`, *Add Input* output của 4 notebook shard (và dataset bảng retrieval nếu có),
   `STAGE = "merge"` → *Save & Run All* → tab Output → *New Dataset*.

In [ ]:
# ===== CẤU HÌNH =====
REPO_URL     = "https://github.com/Giamyy/ms-pred.git"
STRATEGY     = "msg_simulation"   # msg_simulation | msg_known_ce | msg_all
SOURCE_TABLE = None               # None = MassSpecGym1.5.tsv trên Hugging Face (mặc định của script); hoặc đường dẫn tới file đã tải

STAGE        = "shard"            # "shard" | "merge" | "all"
NUM_SHARDS   = 4                  # phải giống nhau ở mọi notebook shard và merge
SHARD_ID     = 0                  # STAGE = "shard": 0 .. NUM_SHARDS-1
SHARD_INPUTS = ["/kaggle/input"]  # STAGE = "merge": nơi tìm output của các notebook shard (tìm đệ quy)

SUBSET_FRAC  = 1.0                # chỉ dùng với STAGE = "all": <1.0 để chạy thử (vd 0.02)
SUBSET_SEED  = 1
MAX_PEAKS    = 50                 # giống data_scripts/dag/run_magma.sh
PPM_DIFF     = 20

# STAGE = "merge"/"all": thư mục (hoặc list thư mục) chứa cands_df_test_formula_256.tsv / cands_df_test_mass_256.tsv.
# None = không thêm bảng retrieval (không ảnh hưởng tới train, chỉ cần cho bước đánh giá retrieval).
RETRIEVAL_DIR = None

In [ ]:
import os, subprocess, sys, json, shutil, time
from pathlib import Path
import pandas as pd

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = Path("/tmp/ms-pred")       # ngoài /kaggle/working: repo và .venv (vài GB) không bị lưu vào output
NCPU = os.cpu_count() or 4
assert STAGE in {"shard", "merge", "all"}, STAGE
if STAGE != "all" and SUBSET_FRAC < 1.0:
    raise ValueError("STAGE shard/merge dùng toàn bộ dữ liệu: đặt SUBSET_FRAC = 1.0")

def sh(cmd, cwd=None, check=True):
    """Chạy lệnh shell, in output ngay khi có; bấm Interrupt sẽ dừng cả tiến trình con."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", start_new_session=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in p.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        os.killpg(p.pid, 9)
        raise
    if p.wait() != 0 and check:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}): {cmd}")
    return p

import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=10)
    print("Internet OK")
except Exception as e:
    raise SystemExit(f"Chưa có internet, hãy bật Settings -> Internet -> On.\n{e}")
print("CPU cores:", NCPU, "| STAGE:", STAGE)

In [ ]:
# ===== 1. Clone repo + cài môi trường =====
if not REPO.exists():
    sh(f"git clone --depth 1 {REPO_URL} {REPO}")
sh("pip install -q uv")
# --python 3.11: Python 3.13 của Kaggle không có wheel torch-scatter / torch-sparse dựng sẵn
for attempt in range(3):          # thử lại nếu mạng chập chờn (download.pytorch.org, GitHub)
    if sh("uv sync --extra cpu --python 3.11", cwd=REPO, check=False).returncode == 0:
        break
    print(f"uv sync lỗi (lần {attempt + 1}/3), thử lại...")
    time.sleep(10)
else:
    raise RuntimeError("uv sync thất bại")
PY = str(REPO / ".venv" / "bin" / "python")
sh(f"{PY} -c \"import sys, ms_pred, rdkit, h5py, torch, dgl, torch_scatter; "
   f"print('python', sys.version.split()[0], '| torch', torch.__version__, '| dgl', dgl.__version__, '| rdkit', rdkit.__version__)\"")

In [ ]:
# ===== 2. Build dataset từ Hugging Face (script chính thức; kết quả giống nhau ở mọi phiên) =====
DS_DIR = REPO / "data" / "spec_datasets" / STRATEGY
src_arg = f"--source-table {SOURCE_TABLE} " if SOURCE_TABLE else ""
sh(f"{PY} data_scripts/create_msg_simulation_dataset.py --strategy {STRATEGY} {src_arg}"
   f"--audit-out data/spec_datasets/{STRATEGY}/audit.json --overwrite", cwd=REPO)
audit = json.load(open(DS_DIR / "audit.json"))
for k in ("source_rows", "output_rows", "output_unique_specs", "unknown_ce_rows", "split_counts", "imputed_check", "pending_resources"):
    print(f"{k:24s}", audit.get(k))
if STRATEGY == "msg_simulation":
    print("README: kỳ vọng 119,029 phổ duy nhất, 0 nhãn imputed (MassSpecGym 1.5).")

In [ ]:
# ===== 3. (Chỉ STAGE = "all") lấy mẫu con theo split để chạy thử =====
DATASET = STRATEGY
if STAGE == "all" and SUBSET_FRAC < 1.0:
    DATASET = f"{STRATEGY}_sub{SUBSET_FRAC:g}"
    SUB = REPO / "data" / "spec_datasets" / DATASET
    (SUB / "splits").mkdir(parents=True, exist_ok=True)
    labels = pd.read_csv(DS_DIR / "labels.tsv", sep="\t")
    splits = pd.read_csv(DS_DIR / "splits" / "split.tsv", sep="\t")
    keep = pd.concat([g.sample(frac=SUBSET_FRAC, random_state=SUBSET_SEED) for _, g in splits.groupby("split")])
    labels = labels[labels["spec"].astype(str).isin(set(keep["name"].astype(str)))]
    labels.to_csv(SUB / "labels.tsv", sep="\t", index=False)
    keep.sort_index().to_csv(SUB / "splits" / "split.tsv", sep="\t", index=False)
    link = SUB / "spec_files.hdf5"
    if link.exists() or link.is_symlink():
        link.unlink()
    link.symlink_to(os.path.relpath(DS_DIR / "spec_files.hdf5", SUB))
    print(DATASET, "->", len(labels), "spectra;", keep["split"].value_counts().to_dict())
else:
    print("Dùng toàn bộ", DATASET)
DATA_DIR = f"data/spec_datasets/{DATASET}"

Hai ô dưới ghi vào repo hai script tách và gộp shard (repo gốc chưa có). Chúng chỉ chia danh sách phổ
và chép kết quả, không đổi nội dung MAGMa / subformula.

In [ ]:
%%writefile /tmp/ms-pred/data_scripts/dag/make_magma_shard.py
#!/usr/bin/env python3
"""Cut one shard out of a dataset so MAGMA can run on it in a separate job.

Shard ``i`` of ``k`` takes rows ``i, i+k, i+2k, ...`` of the source labels.tsv
and copies the matching ``{spec}.ms`` entries of spec_files.hdf5. Run
``bash data_scripts/dag/run_magma.sh <output dataset name>`` on the result,
then combine the shards with data_scripts/dag/merge_magma_shards.py.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import h5py
import pandas as pd


def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--source-dataset", type=Path, required=True,
                        help="Dataset dir with labels.tsv and spec_files.hdf5")
    parser.add_argument("--output-dataset", type=Path, required=True)
    parser.add_argument("--num-shards", type=int, required=True)
    parser.add_argument("--shard-id", type=int, required=True)
    return parser.parse_args()


def main():
    args = get_args()
    if not 0 <= args.shard_id < args.num_shards:
        raise ValueError(f"--shard-id must be in [0, {args.num_shards})")

    labels = pd.read_csv(args.source_dataset / "labels.tsv", sep="\t")
    shard = labels.iloc[args.shard_id::args.num_shards]

    args.output_dataset.mkdir(parents=True, exist_ok=True)
    shard.to_csv(args.output_dataset / "labels.tsv", sep="\t", index=False)

    missing = []
    with h5py.File(args.source_dataset / "spec_files.hdf5", "r") as src, \
            h5py.File(args.output_dataset / "spec_files.hdf5", "w") as dst:
        for spec in shard["spec"].astype(str):
            key = f"{spec}.ms"
            if key in src:
                src.copy(key, dst, name=key)
            else:
                missing.append(spec)

    summary = {
        "source_specs": int(len(labels)),
        "shard_id": args.shard_id,
        "num_shards": args.num_shards,
        "shard_specs": int(len(shard)),
        "missing_in_spec_files": len(missing),
    }
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /tmp/ms-pred/data_scripts/dag/merge_magma_shards.py
#!/usr/bin/env python3
"""Merge MAGMA and subformula outputs computed on dataset shards.

Each shard dir (see make_magma_shard.py) holds
``magma_outputs/magma_tree.hdf5`` and ``subformulae/no_subform.hdf5``. Shards
cover disjoint spectra, so merging copies every top-level entry into one file:

* magma_tree.hdf5 is a PredSpecDB: root attributes are copied from the first
  shard (all shards must agree) and the embedded manifest is rebuilt from the
  merged groups.
* no_subform.hdf5 holds one ``{spec}_collision {ce}.json`` string per entry.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import h5py

from ms_pred.common.misc_utils import (
    _MANIFEST_GROUP,
    _build_manifest_from_old_structure,
    _embed_manifest_into_h5,
)

MAGMA_REL = Path("magma_outputs/magma_tree.hdf5")
SUBFORM_REL = Path("subformulae/no_subform.hdf5")


def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--shard-dirs", type=Path, nargs="+", required=True)
    parser.add_argument("--output-dir", type=Path, required=True,
                        help="Dataset dir that receives magma_outputs/ and subformulae/")
    parser.add_argument("--overwrite", action="store_true")
    return parser.parse_args()


def copy_entries(shard_files: list[Path], out_path: Path, skip: set[str]) -> dict:
    written, duplicates = 0, []
    with h5py.File(out_path, "a") as dst:
        for path in shard_files:
            with h5py.File(path, "r") as src:
                for key in src.keys():
                    if key in skip:
                        continue
                    if key in dst:
                        duplicates.append(key)
                        continue
                    src.copy(key, dst, name=key)
                    written += 1
    if duplicates:
        raise ValueError(f"{len(duplicates)} entries appear in more than one shard, e.g. {duplicates[:5]}")
    return {"entries": written}


def merge_magma(shard_files: list[Path], out_path: Path) -> dict:
    with h5py.File(shard_files[0], "r") as first:
        root_attrs = dict(first.attrs)
    for path in shard_files[1:]:
        with h5py.File(path, "r") as f:
            if dict(f.attrs) != root_attrs:
                raise ValueError(f"Root attributes of {path} differ from {shard_files[0]}")

    with h5py.File(out_path, "w") as dst:
        dst.attrs.update(root_attrs)
    summary = copy_entries(shard_files, out_path, skip={_MANIFEST_GROUP})

    with h5py.File(out_path, "a") as dst:
        manifest = _build_manifest_from_old_structure(dst)
        _embed_manifest_into_h5(dst, manifest, out_path)
    summary["manifest_entries"] = len(manifest["leaf_path"])
    return summary


def merge_subformulae(shard_files: list[Path], out_path: Path) -> dict:
    with h5py.File(out_path, "w"):
        pass
    return copy_entries(shard_files, out_path, skip=set())


def main():
    args = get_args()

    outputs = {MAGMA_REL: args.output_dir / MAGMA_REL, SUBFORM_REL: args.output_dir / SUBFORM_REL}
    for rel in outputs:
        missing = [str(d) for d in args.shard_dirs if not (d / rel).exists()]
        if missing:
            raise FileNotFoundError(f"{rel} missing in shard dirs: {missing}")
    for out in outputs.values():
        if out.exists() and not args.overwrite:
            raise FileExistsError(f"{out} exists; pass --overwrite to replace it")
        out.parent.mkdir(parents=True, exist_ok=True)

    summary = {
        "shards": [str(d) for d in args.shard_dirs],
        "magma_tree": merge_magma([d / MAGMA_REL for d in args.shard_dirs], outputs[MAGMA_REL]),
        "no_subform": merge_subformulae([d / SUBFORM_REL for d in args.shard_dirs], outputs[SUBFORM_REL]),
    }
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
# ===== 4. MAGMa + subformula (CPU, bước lâu nhất) =====
def magma_and_subform(data_dir):
    # Giống data_scripts/dag/run_magma.sh: MAGMa rồi subformula --use-all
    sh(f"{PY} src/ms_pred/magma/run_magma.py --spectra-dir {data_dir}/spec_files.hdf5 "
       f"--output-dir {data_dir}/magma_outputs --spec-labels {data_dir}/labels.tsv "
       f"--max-peaks {MAX_PEAKS} --ppm-diff {PPM_DIFF} --workers {NCPU}", cwd=REPO)
    (REPO / data_dir / "subformulae").mkdir(parents=True, exist_ok=True)
    sh(f"{PY} data_scripts/forms/01_assign_subformulae.py --data-dir {data_dir} "
       f"--labels-file {data_dir}/labels.tsv --use-all --output-dir-name no_subform.hdf5 --num-workers {NCPU}", cwd=REPO)

t0 = time.time()
if STAGE == "all":
    magma_and_subform(DATA_DIR)

elif STAGE == "shard":
    SHARD_DIR = f"data/spec_datasets/{STRATEGY}_shard{SHARD_ID}"
    sh(f"{PY} data_scripts/dag/make_magma_shard.py --source-dataset {DATA_DIR} --output-dataset {SHARD_DIR} "
       f"--num-shards {NUM_SHARDS} --shard-id {SHARD_ID}", cwd=REPO)
    magma_and_subform(SHARD_DIR)
    (REPO / SHARD_DIR / "spec_files.hdf5").unlink()      # phiên merge tự build lại, không cần lưu
    (REPO / SHARD_DIR / "shard.json").write_text(json.dumps(
        {"strategy": STRATEGY, "shard_id": SHARD_ID, "num_shards": NUM_SHARDS, "max_peaks": MAX_PEAKS, "ppm_diff": PPM_DIFF}))

elif STAGE == "merge":
    found = sorted({p.parents[1] for root in SHARD_INPUTS
                    for p in Path(root).glob(f"**/{STRATEGY}_shard*/magma_outputs/magma_tree.hdf5")})
    for d in found:
        print("Tìm thấy:", d)
    metas = [json.loads((d / "shard.json").read_text()) for d in found]
    if sorted(m["shard_id"] for m in metas) != list(range(NUM_SHARDS)) or \
            any(m["num_shards"] != NUM_SHARDS or m["strategy"] != STRATEGY for m in metas):
        raise RuntimeError(f"Cần đủ {NUM_SHARDS} shard (0..{NUM_SHARDS - 1}) của {STRATEGY}, tìm thấy {metas}")
    if any((m["max_peaks"], m["ppm_diff"]) != (MAX_PEAKS, PPM_DIFF) for m in metas):
        raise RuntimeError("Các shard chạy với MAX_PEAKS / PPM_DIFF khác nhau")
    covered = set().union(*(set(pd.read_csv(d / "labels.tsv", sep="\t")["spec"].astype(str)) for d in found))
    expected = set(pd.read_csv(DS_DIR / "labels.tsv", sep="\t")["spec"].astype(str))
    if covered != expected:
        raise RuntimeError(f"Các shard phủ {len(covered)} phổ, labels.tsv có {len(expected)}")
    sh(f"{PY} data_scripts/dag/merge_magma_shards.py --shard-dirs {' '.join(map(str, found))} "
       f"--output-dir {DS_DIR} --overwrite", cwd=REPO)
print(f"Bước 4: {(time.time() - t0) / 60:.1f} phút")

In [ ]:
# ===== 5. (Tuỳ chọn) bảng ứng viên retrieval, lọc theo phổ của dataset như script chính thức =====
if STAGE != "shard" and RETRIEVAL_DIR:
    dirs = RETRIEVAL_DIR if isinstance(RETRIEVAL_DIR, (list, tuple)) else [RETRIEVAL_DIR]
    dirs = [Path(d).parent if Path(d).is_file() else Path(d) for d in dirs]
    specs = set(pd.read_csv(REPO / DATA_DIR / "labels.tsv", sep="\t")["spec"].astype(str))
    out = REPO / DATA_DIR / "retrieval"
    out.mkdir(exist_ok=True)
    for name in ["cands_df_test_formula_256.tsv", "cands_df_test_mass_256.tsv"]:
        src = next((d / name for d in dirs if (d / name).exists()), None)
        if src is None:
            print("Không có", name, "trong", [str(d) for d in dirs])
            continue
        df = pd.concat(c[c["spec"].astype(str).isin(specs)] for c in pd.read_csv(src, sep="\t", chunksize=100000))
        if "collision_energies" in df:
            df = df[~df["collision_energies"].astype(str).str.contains(r"\[imputed\]", regex=True)]
        df.to_csv(out / name, sep="\t", index=False)
        print(f"{name}: {len(df)} dòng, {df['spec'].nunique()} phổ")
else:
    print("Bỏ qua bảng retrieval")

In [ ]:
# ===== 6. Kiểm tra =====
import h5py
check_dir = REPO / (SHARD_DIR if STAGE == "shard" else DATA_DIR)
labels = pd.read_csv(check_dir / "labels.tsv", sep="\t")
n = len(labels)
with h5py.File(check_dir / "magma_outputs/magma_tree.hdf5") as f:
    n_magma = len([k for k in f if not k.startswith("__")])
with h5py.File(check_dir / "subformulae/no_subform.hdf5") as f:
    n_sub = len(f)
print(f"{check_dir.name}: labels {n} | magma {n_magma} ({n_magma / n:.1%}) | subformula {n_sub}")
assert n_magma / n > 0.99 and n_sub == n, "Thiếu kết quả MAGMa/subformula"
if STAGE != "shard":
    for rel in ["splits/split.tsv", "spec_files.hdf5", "audit.json"]:
        assert (check_dir / rel).exists(), rel
    print("SẴN SÀNG cho MARASON" + ("" if STAGE == "all" and SUBSET_FRAC < 1.0 else f" ({n} phổ)"))

In [ ]:
# ===== 7. Đóng gói vào /kaggle/working =====
src = REPO / (SHARD_DIR if STAGE == "shard" else DATA_DIR)
out = WORK / (src.name if STAGE == "shard" else f"{DATASET}_processed")
if out.exists():
    shutil.rmtree(out)
shutil.copytree(src, out, symlinks=False)     # symlinks=False: chép file thật thay cho symlink
sh(f"du -sh {out}/* | sort -h")
print("Xong:", out)
print("Bấm Save Version (Save & Run All) để lưu." + ("" if STAGE == "shard" else " Rồi tab Output -> New Dataset."))

## Dùng lại ở notebook train

Gắn dataset đã lưu vào notebook train, rồi chép về đúng chỗ repo mong đợi:

```python
!mkdir -p ms-pred/data/spec_datasets/msg_simulation
!cp -r /kaggle/input/<ten-dataset>/msg_simulation_processed/* ms-pred/data/spec_datasets/msg_simulation/
```

Các bước train MARASON và các chỗ cần sửa đường dẫn: xem `notebooks/kaggle/HUONG_DAN.md`, Phần B.
Train MARASON theo README cần GPU lớn; trên Kaggle (T4 16GB) cần giảm `batch-size` và `num-workers`.